### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="mutual_funds_india",
    dataset_year="2023",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/ravibarnawal/mutual-funds-india-detailed",
    download_description="""
kaggle datasets download ravibarnawal/mutual-funds-india-detailed --unzip && mkdir -p local-data-warehouse/mutual_funds_india && mv comprehensive_mutual_funds_data.csv local-data-warehouse/mutual_funds_india/
""",
    # References
    academic_reference_bibtex=r"""@misc{Barnawal2022MutualFundsIndiaDetailed,
  author = {Ravi Barnawal},
  title  = {Mutual Funds India Detailed},
  year   = {2022},
  howpublished = {\url{https://www.kaggle.com/datasets/ravibarnawal/mutual-funds-india-detailed}},
  note   = {Kaggle dataset}
}
""",
    academic_reference_bibtex_key="Barnawal2022MutualFundsIndiaDetailed",
    license="CC0: Public Domain",
    data_tags=["IID"],
    curation_comments="""
We start with the data from Kaggle.

- We aim to predict the return over 3 years and drop all samples that do not have data for 3 year return.
- We drop potentially leaking columns about returns (alpha, beta, sd, sharpe, sortino).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="returns_3yr",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "comprehensive_mutual_funds_data.csv")
print("Loaded data shape:", df.shape)

df = df.drop(columns=["alpha", "beta","sd", "sharpe", "sortino", "returns_5yr", "returns_1yr"])

df = df[df["returns_3yr"].notna()]

as_string_type = [
    "scheme_name",
    "amc_name",
    "fund_manager",
]
as_cat_type = [
    "sub_category",
    "category",
]
for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (814, 20)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 793
Columns: 13
Use sampling: False (sample size: 793)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['scheme_name', 'fund_size_cr', 'fund_manager', 'expense_ratio', 'amc_name', 'sub_category', 'fund_age_yr', 'min_lumpsum', 'min_sip', 'risk_level']
Rows remaining as candidates after top-10 filter: 0 (of 793)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,scheme_name,min_sip,min_lumpsum,expense_ratio,fund_size_cr,fund_age_yr,fund_manager,risk_level,amc_name,rating,category,sub_category,returns_3yr
0,Canara Robeco Savings Fund,1000,5000,0.31,1003.0,10,Kunal Jain,2,Canara Robeco Mutual Fund,2,Debt,Low Duration Funds,5.2
1,Franklin India Debt Hybrid Fund,500,10000,0.57,246.0,10,Rajasa Kakulavarapu,4,Franklin Templeton Mutual Fund,3,Hybrid,Conservative Hybrid Mutual Funds,9.8
2,Union Arbitrage Fund,500,1000,0.44,72.0,4,Vishal Thakker,1,Union Mutual Fund,3,Hybrid,Arbitrage Mutual Funds,4.4
3,PGIM India Overnight Fund,1000,100,0.12,75.0,4,Puneet Pal,1,PGIM India Mutual Fund,4,Debt,Overnight Mutual Funds,3.9
4,Sundaram Aggressive Hybrid Fund,100,100,0.67,2981.0,10,S Bharath,5,Sundaram Mutual Fund,4,Hybrid,Aggressive Hybrid Mutual Funds,21.7


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,category,category,0.0,0.0,5.0,"Equity, Debt, Hybrid, Other, Solution Oriented"
1,sub_category,category,0.0,0.0,38.0,"Sectoral / Thematic Mutual Funds, ELSS Mutual Funds, FoFs Domestic, Liquid Mutual Funds, Aggressive Hybrid Mutual Funds, Index Funds, Large Cap Mutual Funds, Overnight Mutual Funds, Small Cap Mutual Funds, Large & Mid Cap Funds"
2,expense_ratio,float64,0.0,0.0,177.0,"0.21, 0.22, 0.15, 0.39, 0.1, 0.2, 0.28, 0.4, 1.0, 0.37"
3,fund_size_cr,float64,0.0,0.0,683.0,"90.0, 30.0, 26.0, 49.0, 36.0, 1699.0, 91.0, 141.0, 34.0, 27.0"
4,returns_3yr,float64,0.0,0.0,304.0,"3.9, 4.4, 5.6, 4.3, 5.1, 32.4, 4.9, 6.9, 6.2, 25.0"
5,min_sip,int64,0.0,0.0,10.0,"500, 1000, 100, 0, 150, 300, 2000, 250, 99, 750"
6,min_lumpsum,int64,0.0,0.0,11.0,"5000, 500, 1000, 100, 10000, 10, 0, 15000, 99, 25000"
7,fund_age_yr,int64,0.0,0.0,14.0,"10, 4, 5, 9, 8, 3, 7, 6, 14, 17"
8,risk_level,int64,0.0,0.0,6.0,"6, 3, 2, 4, 1, 5"
9,rating,int64,0.0,0.0,6.0,"3, 2, 4, 0, 5, 1"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
min_sip,793.0,522.634300,366.325664,0.00,2000.00
min_lumpsum,793.0,3042.924338,2525.267537,0.00,25000.00
expense_ratio,793.0,0.724741,0.482604,0.00,2.59
fund_size_cr,793.0,3898.649849,7254.601758,2.38,57052.00
fund_age_yr,793.0,8.482976,2.473093,1.00,17.00
risk_level,793.0,4.453972,1.802275,1.00,6.00
rating,793.0,2.636822,1.471705,0.00,5.00
returns_3yr,793.0,18.525347,12.108476,3.30,71.40


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column       rank                                                 
amc_name     1          ICICI Prudential Mutual Fund     57   7.19
             2     Aditya Birla Sun Life Mutual Fund     49   6.18
             3                       SBI Mutual Fund     47   5.93
             4                  Sundaram Mutual Fund     41   5.17
             5              Nippon India Mutual Fund     39   4.92
category     1                                Equity    307  38.71
             2                                  Debt    263  33.17
             3                                Hybrid    116  14.63
             4                                 Other     79   9.96
             5                     Solution Oriented     28   3.53
fund_manager 1                        Rohit Seksaria     18   2.27
             2                        Deepak Agrawal     12   1.51
             3                          R Srinivasan     12   1.51
             4                        Manish Banthia     11   1.39
             5                           Devang Shah     11   1.39
scheme_name  1          SBI Long Term Advantage Fund      6   0.76
             2             ICICI Pru Retirement Fund      4    0.5
             3          HDFC Retirement Savings Fund      3   0.38
             4          AXIS Retirement Savings Fund      3   0.38
             5          Tata Retirement Savings Fund      3   0.38
sub_category 1      Sectoral / Thematic Mutual Funds     82  10.34
             2                     ELSS Mutual Funds     52   6.56
             3                         FoFs Domestic     34   4.29
             4                   Liquid Mutual Funds     33   4.16
             5        Aggressive Hybrid Mutual Funds     30   3.78

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.479,-0.295,146.615,0.621,log,6250.4,6293.8,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to mutual_funds_india/019d5dcd-9ed5-7c7c-b000-801f0b368948


019d5dcd-9ed5-7c7c-b000-801f0b368948
b999916b62a91414b5bf6ca39cd213e9cb3ae8f566e3a8b2e23756169a61d8a4
